# FEM: 2D Laplace Solver — Griffiths Gap Filler

Griffiths solves EM analytically on symmetric geometries.  
FEM solves it numerically on **any** geometry — CAD, MRI coils, chip interconnects.

Same equation:  `∇²φ = 0`  (Laplace) or  `∇²φ = -ρ/ε₀`  (Poisson)

This notebook: finite difference method (simplest FEM) on a 2D grid.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import sympy as sp
sp.init_printing(use_unicode=False)
print('imports OK')

## S1. Laplace equation: the physics

Griffiths 3.1: any charge-free region satisfies Laplace's equation.

The 5-point finite difference stencil:
```
∇²φ ≈ (φ[i+1,j] + φ[i-1,j] + φ[i,j+1] + φ[i,j-1] - 4φ[i,j]) / h²  = 0
=> φ[i,j] = (φ[i+1,j] + φ[i-1,j] + φ[i,j+1] + φ[i,j-1]) / 4
```
Every interior point = average of its 4 neighbors. Iterate until convergence.

In [ ]:
# S1: SymPy derivation of finite difference
h, phi0, phi1, phi2, phi3, phi4 = sp.symbols('h phi_0 phi_1 phi_2 phi_3 phi_4')

# Taylor expand phi(x+h) and phi(x-h)
x = sp.Symbol('x')
f = sp.Function('f')
taylor_p = sp.series(f(x+h), h, 0, 5)
taylor_m = sp.series(f(x-h), h, 0, 5)
d2 = sp.simplify((taylor_p + taylor_m - 2*f(x)) / h**2)
print('d^2f/dx^2 from Taylor expansion:')
print(sp.pretty(d2))
print('Leading term: f(x+h) + f(x-h) - 2f(x)) / h^2  + O(h^2)')

## S2. Solve: parallel plate capacitor

Boundary conditions:
- Top wall: φ = +1 V  
- Bottom wall: φ = -1 V  
- Left/right: φ = 0 (grounded sides)

Analytical: φ(y) = y/L — linear between plates.  
FEM: should match exactly.

In [ ]:
def solve_laplace(N=64, bc_top=1.0, bc_bot=-1.0, bc_left=0.0, bc_right=0.0,
                  n_iter=2000, tol=1e-6, obstacle_mask=None):
    """Finite difference Laplace solver. obstacle_mask: 2D bool array of fixed-phi regions."""
    phi = np.zeros((N, N))
    # boundary conditions
    phi[0, :]  = bc_bot
    phi[-1, :] = bc_top
    phi[:, 0]  = bc_left
    phi[:, -1] = bc_right

    for iteration in range(n_iter):
        phi_old = phi.copy()
        # vectorized Jacobi update
        phi[1:-1, 1:-1] = 0.25 * (
            phi[2:, 1:-1] + phi[:-2, 1:-1] +
            phi[1:-1, 2:] + phi[1:-1, :-2]
        )
        # restore boundaries
        phi[0, :]  = bc_bot
        phi[-1, :] = bc_top
        phi[:, 0]  = bc_left
        phi[:, -1] = bc_right
        # restore obstacle (fixed potential)
        if obstacle_mask is not None:
            phi[obstacle_mask] = 0.0
        # convergence check
        if np.max(np.abs(phi - phi_old)) < tol:
            print(f'  Converged at iteration {iteration}')
            break
    return phi

N = 64
phi_cap = solve_laplace(N=N, bc_top=1.0, bc_bot=-1.0, bc_left=0.0, bc_right=0.0)

# compare to analytical
y = np.linspace(-1, 1, N)
phi_analytical = np.tile(y, (N, 1)).T
print(f'Max error vs analytical: {np.max(np.abs(phi_cap - phi_analytical)):.2e}')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Parallel Plate Capacitor: FEM vs Analytical', fontweight='bold')

im0 = axes[0].imshow(phi_cap, origin='lower', cmap='RdBu', vmin=-1, vmax=1)
axes[0].set_title('FEM solution phi(x,y)')
plt.colorbar(im0, ax=axes[0], label='V')

im1 = axes[1].imshow(phi_analytical, origin='lower', cmap='RdBu', vmin=-1, vmax=1)
axes[1].set_title('Analytical phi = y/L')
plt.colorbar(im1, ax=axes[1], label='V')

err = np.abs(phi_cap - phi_analytical)
im2 = axes[2].imshow(err, origin='lower', cmap='hot')
axes[2].set_title('|Error|')
plt.colorbar(im2, ax=axes[2], label='V')

plt.tight_layout()
plt.show()

## S3. Electric field E = -grad(phi)

In [ ]:
# gradient via numpy
Ey, Ex = np.gradient(-phi_cap)

fig, ax = plt.subplots(1, 1, figsize=(7, 6))
ax.imshow(phi_cap, origin='lower', cmap='RdBu', alpha=0.6, vmin=-1, vmax=1)

# quiver every 4 points
step = 4
Y, X = np.mgrid[0:N:step, 0:N:step]
ax.quiver(X, Y, Ex[::step, ::step], Ey[::step, ::step],
          color='k', scale=20, width=0.003)
ax.set_title('E = -grad(phi): field lines in capacitor')
ax.set_xlabel('x'); ax.set_ylabel('y')
plt.tight_layout()
plt.show()

print(f'Mean |E_y| (should be ~1/L=1): {np.mean(np.abs(Ey[1:-1,1:-1])):.4f}')
print(f'Mean |E_x| (should be ~0):     {np.mean(np.abs(Ex[1:-1,1:-1])):.4f}')

## S4. Irregular geometry: conductor obstacle

This is what Griffiths **cannot** do analytically.  
FEM handles it trivially — just fix phi=0 inside the obstacle.

In [ ]:
N = 64
# square conductor obstacle in center
mask = np.zeros((N, N), dtype=bool)
mask[N//3:2*N//3, N//3:2*N//3] = True

phi_obs = solve_laplace(N=N, bc_top=1.0, bc_bot=-1.0,
                        bc_left=0.0, bc_right=0.0,
                        obstacle_mask=mask, n_iter=5000)
phi_obs[mask] = 0.0

Ey_o, Ex_o = np.gradient(-phi_obs)
E_mag = np.sqrt(Ex_o**2 + Ey_o**2)
E_mag[mask] = 0  # zero inside conductor

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Conductor obstacle in capacitor field', fontweight='bold')

im0 = axes[0].imshow(phi_obs, origin='lower', cmap='RdBu', vmin=-1, vmax=1)
axes[0].set_title('phi(x,y) with conductor')
plt.colorbar(im0, ax=axes[0], label='V')

im1 = axes[1].imshow(E_mag, origin='lower', cmap='hot')
axes[1].set_title('|E| field magnitude (hot = strong)')
plt.colorbar(im1, ax=axes[1], label='V/m')

step = 4
Y, X = np.mgrid[0:N:step, 0:N:step]
axes[0].quiver(X, Y, Ex_o[::step,::step], Ey_o[::step,::step],
               color='k', scale=20, width=0.003, alpha=0.7)

plt.tight_layout()
plt.show()
print('Field concentrates at edges of conductor (edge enhancement = capacitor fringe effect)')

## S5. Poisson equation: space charge

`∇²φ = -ρ/ε₀`  — point charge in a grounded box.

In [ ]:
def solve_poisson(N, rho, eps0=1.0, n_iter=5000, tol=1e-6):
    """Finite difference Poisson solver. rho: 2D charge density array."""
    h = 1.0 / N
    phi = np.zeros((N, N))
    src = -rho * h**2 / eps0
    for iteration in range(n_iter):
        phi_old = phi.copy()
        phi[1:-1,1:-1] = 0.25 * (
            phi[2:,1:-1] + phi[:-2,1:-1] +
            phi[1:-1,2:] + phi[1:-1,:-2]
            - src[1:-1,1:-1]
        )
        phi[0,:] = phi[-1,:] = phi[:,0] = phi[:,-1] = 0.0
        if np.max(np.abs(phi - phi_old)) < tol:
            print(f'  Converged at iteration {iteration}')
            break
    return phi

N = 64
rho = np.zeros((N, N))
# positive charge at center-left, negative at center-right
rho[N//2, N//3]   = +100.0
rho[N//2, 2*N//3] = -100.0

phi_p = solve_poisson(N, rho)
Ey_p, Ex_p = np.gradient(-phi_p)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(phi_p, origin='lower', cmap='RdBu')
plt.colorbar(im, ax=ax, label='V')
step = 4
Y, X = np.mgrid[0:N:step, 0:N:step]
ax.quiver(X, Y, Ex_p[::step,::step], Ey_p[::step,::step],
          color='k', scale=30, width=0.003)
ax.plot(N//3, N//2, 'r+', ms=15, mew=3, label='+q')
ax.plot(2*N//3, N//2, 'b+', ms=15, mew=3, label='-q')
ax.legend()
ax.set_title('Poisson: dipole in grounded box (Griffiths 3.3 numerically)')
plt.tight_layout()
plt.show()

## S6. Connection to GS phase recovery

Same mathematical structure:

| Problem | Equation | Iterative solver |
|---------|----------|------------------|
| Laplace FEM | `phi[i,j] = avg(neighbors)` | Jacobi iteration |
| GS phase recovery | `E = P_C2(P_C1(E))` | Alternating projections |
| Attention | `out = softmax(QK^T/sqrt(d)) V` | Soft projection |

All three: **find the fixed point of an iterative projection.**  
FEM converges because Laplace operator is positive definite.  
GS converges when constraint sets intersect (diversity |ΔD| is large).

In [ ]:
# Convergence rate comparison
import time

sizes = [32, 64, 128]
print(f'{'N':>6}  {'iters':>8}  {'time_ms':>10}  {'max_err':>10}')
for N_s in sizes:
    rho_s = np.zeros((N_s, N_s))
    rho_s[N_s//2, N_s//2] = 100.0
    t0 = time.perf_counter()
    phi_s = solve_poisson(N_s, rho_s, n_iter=10000, tol=1e-5)
    ms = (time.perf_counter()-t0)*1000
    print(f'{N_s:>6}  {'done':>8}  {ms:>10.1f}  {np.max(np.abs(phi_s)):>10.4f}')